# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata fields
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's enumerate the record sets, display their `@id` and their associated field `@id`s.

In [ ]:
# Get and print all record set IDs and their fields
record_sets = dataset.record_sets
print(f"Record sets found: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"Record set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {record_set.description}\n")
    print("  Field @ids:")
    for field in record_set.fields:
        print(f"    - {field.id} ({field.name})")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we extract data from all record sets found in the overview above. You can adjust the selected record set for your particular analysis by referencing the record set's `@id`.

In [ ]:
# Extract data from all record sets
record_set_ids = [rset.id for rset in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records):
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print("  No records found.")
print("")
# Example: Show columns and a sample from the first record set
if record_set_ids:
    chosen_set = record_set_ids[0]
    print(f"Columns for record set '{chosen_set}':")
    print(dataframes[chosen_set].columns.tolist())
    dataframes[chosen_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. Use the correct field `@id`s as shown in the overview section above.

In [ ]:
# Edit the following to use your chosen record_set_id and field ids from above
# If you only have one record set, this will pick the first automatically
if dataframes:
    record_set_id = list(dataframes.keys())[0]  # Replace with actual @id if needed
    df = dataframes[record_set_id]

    # Example: pick a numeric field
    numeric_field_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    print(f"Numeric fields (candidates): {numeric_field_candidates}")
    # For demonstration, pick the first numeric field
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the chosen numeric field
        filtered_df.loc[:, f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping example
        # Look for a non-numeric field for grouping:
        group_field_candidates = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
        print(f"Group field candidates: {group_field_candidates}")
        if group_field_candidates:
            group_field = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found in the record set to demonstrate EDA.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Adjust field names for your specific dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sample visualization: histogram of numeric field and countplot of a categorical field
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id].copy()
    numeric_columns = [c for c in df.columns if df[c].dtype in ('int64', 'float64')]
    object_columns = [c for c in df.columns if df[c].dtype == 'object']

    if numeric_columns:
        # Plot distribution of first numeric field
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_columns[0]].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_columns[0]}")
        plt.xlabel(numeric_columns[0])
        plt.show()
    if len(numeric_columns) > 1:
        # Scatter plot of the first two numeric fields
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=df[numeric_columns[0]], y=df[numeric_columns[1]])
        plt.xlabel(numeric_columns[0])
        plt.ylabel(numeric_columns[1])
        plt.title(f"Scatter plot: {numeric_columns[0]} vs {numeric_columns[1]}")
        plt.show()
    if object_columns:
        # Plot categorical count
        plt.figure(figsize=(8, 4))
        sns.countplot(y=df[object_columns[0]])
        plt.title(f"Counts by {object_columns[0]}")
        plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using `mlcroissant`.

- **Metadata access**: Easily inspect name and description
- **Record set and fields overview**: Quickly review available structures with their `@id`s
- **Data extraction**: Loaded data into DataFrames for each record set using their `@id`
- **EDA & Visualization**: Demonstrated basic cleaning, normalization, grouping, and created visualizations

You can now customize this notebook further to perform deeper analyses depending on your research questions.

<sub>Notebook generated via automated instructions for transparent, reproducible FAIR data analysis.</sub>